In [2]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

# Grouping by day
Create 7-day activity vectors for the first week of September 2024 (Sept 1-7).

In [3]:
# Initialize DuckDB connection
con = duckdb.connect()

# Load users of interest (users who blocked in first week of Sept 2024)
user_of_interests = "../../data/ale_simplicistic_model/absolute/filtered/first_week_sept_blockers.parquet"
con.execute("CREATE TABLE users_of_interest AS SELECT * FROM read_parquet(?)", [user_of_interests])

print("Users of interest loaded")

Users of interest loaded


In [4]:
# Helper: build 7-day vectors using DuckDB for efficient large-table processing
# Days are: Sept 1 (day 0), Sept 2 (day 1), ..., Sept 7 (day 6)
def make_vec_duckdb(file_path, left_on='did_id', vec_name='vec'):
    """
    Use DuckDB to:
    1. Read parquet file
    2. Compute day of September (1-7 maps to 0-6)
    3. Aggregate counts per user per day
    4. Pivot to create 7-element vectors
    """
    query = f"""
    WITH activity AS (
        SELECT 
            act.{left_on} as id_col,
            act.created_date
        FROM read_parquet('{file_path}') act
        WHERE act.created_date >= '2024-09-01' 
          AND act.created_date < '2024-09-08'
    ),
    days_computed AS (
        SELECT 
            id_col,
            CAST(DATE_DIFF('day', DATE '2024-09-01', created_date::DATE) AS INTEGER) AS day_index
        FROM activity
    ),
    counts AS (
        SELECT 
            id_col,
            day_index,
            COUNT(*) as cnt
        FROM days_computed
        WHERE day_index >= 0 AND day_index <= 6
        GROUP BY id_col, day_index
    ),
    pivoted AS (
        SELECT
            id_col,
            MAX(CASE WHEN day_index = 0 THEN cnt ELSE 0 END) AS d0,
            MAX(CASE WHEN day_index = 1 THEN cnt ELSE 0 END) AS d1,
            MAX(CASE WHEN day_index = 2 THEN cnt ELSE 0 END) AS d2,
            MAX(CASE WHEN day_index = 3 THEN cnt ELSE 0 END) AS d3,
            MAX(CASE WHEN day_index = 4 THEN cnt ELSE 0 END) AS d4,
            MAX(CASE WHEN day_index = 5 THEN cnt ELSE 0 END) AS d5,
            MAX(CASE WHEN day_index = 6 THEN cnt ELSE 0 END) AS d6
        FROM counts
        GROUP BY id_col
    )
    SELECT 
        id_col AS did_id,
        [d0, d1, d2, d3, d4, d5, d6] AS {vec_name}
    FROM pivoted
    """
    
    return con.execute(query).df()

print("DuckDB helper function defined")

DuckDB helper function defined


In [5]:
# Posts vectors using DuckDB
posts_path = "../../data/ale_simplicistic_model/absolute/filtered/posts.parquet"
posts_vec_df = make_vec_duckdb(posts_path, left_on='did_id', vec_name='posts_vec')

# Register the pandas DataFrame so DuckDB can reference it by name
con.register("posts_vec_df", posts_vec_df)

# Create uoi_activity with posts_vec already COALESCED to zeros when missing
con.execute("""
CREATE TABLE uoi_activity AS
SELECT uoi.*,
       COALESCE(pv.posts_vec, [0,0,0,0,0,0,0]) AS posts_vec
FROM users_of_interest uoi
LEFT JOIN posts_vec_df pv ON uoi.did_id = pv.did_id
""")

print("Posts processed:")
print(con.execute("SELECT did_id, posts_vec FROM uoi_activity LIMIT 5").df())

Posts processed:
     did_id                            posts_vec
0  18953466               [1, 4, 14, 7, 4, 8, 2]
1    181936  [386, 352, 295, 195, 189, 378, 154]
2  11911092          [52, 17, 55, 29, 11, 2, 16]
3   3888430          [33, 39, 24, 27, 19, 9, 46]
4  19820116          [14, 15, 18, 10, 10, 16, 9]


In [6]:
# Blocks: actor and subject 7-day vectors using DuckDB
blocks_path = "../../data/ale_simplicistic_model/absolute/filtered/blocks.parquet"

# Actor vectors
blocks_actor_vec = make_vec_duckdb(blocks_path, left_on='did_id', vec_name='blocks_actor_vec')
con.register("blocks_actor_vec", blocks_actor_vec)
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(bav.blocks_actor_vec, [0,0,0,0,0,0,0]) AS blocks_actor_vec
    FROM uoi_activity ua
    LEFT JOIN blocks_actor_vec bav ON ua.did_id = bav.did_id
""")

# Subject vectors
blocks_subject_vec = make_vec_duckdb(blocks_path, left_on='subject_did_id', vec_name='blocks_subject_vec')
con.register("blocks_subject_vec", blocks_subject_vec)
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(bsv.blocks_subject_vec, [0,0,0,0,0,0,0]) AS blocks_subject_vec
    FROM uoi_activity ua
    LEFT JOIN blocks_subject_vec bsv ON ua.did_id = bsv.did_id
""")

print("Blocks processed:")
print(con.execute("SELECT did_id, blocks_actor_vec, blocks_subject_vec FROM uoi_activity LIMIT 5").df())

Blocks processed:
     did_id        blocks_actor_vec     blocks_subject_vec
0  14897458   [0, 0, 0, 1, 3, 0, 0]  [0, 0, 0, 1, 0, 0, 1]
1   9513897   [0, 0, 0, 1, 0, 0, 0]  [0, 0, 1, 0, 0, 0, 1]
2  11790358  [0, 17, 0, 0, 0, 0, 0]  [0, 0, 0, 1, 0, 0, 0]
3  28137121   [0, 0, 0, 0, 1, 0, 0]  [0, 0, 0, 0, 0, 1, 0]
4   7346482   [0, 6, 0, 0, 0, 0, 0]  [0, 0, 0, 0, 1, 0, 0]


In [7]:
# Follows: actor and subject vectors using DuckDB (handles large tables efficiently)
follows_path = "../../data/ale_simplicistic_model/absolute/filtered/follows.parquet"

# Actor vectors
follows_actor_vec = make_vec_duckdb(follows_path, left_on='did_id', vec_name='follows_actor_vec')
con.register("follows_actor_vec", follows_actor_vec)
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(fav.follows_actor_vec, [0,0,0,0,0,0,0]) AS follows_actor_vec
    FROM uoi_activity ua
    LEFT JOIN follows_actor_vec fav ON ua.did_id = fav.did_id
""")

# Subject vectors
follows_subject_vec = make_vec_duckdb(follows_path, left_on='subject_did_id', vec_name='follows_subject_vec')
con.register("follows_subject_vec", follows_subject_vec)
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(fsv.follows_subject_vec, [0,0,0,0,0,0,0]) AS follows_subject_vec
    FROM uoi_activity ua
    LEFT JOIN follows_subject_vec fsv ON ua.did_id = fsv.did_id
""")

print("Follows processed:")
print(con.execute("SELECT did_id, follows_actor_vec, follows_subject_vec FROM uoi_activity LIMIT 5").df())

Follows processed:
     did_id           follows_actor_vec           follows_subject_vec
0   1928789  [39, 11, 9, 3, 2, 135, 11]     [35, 12, 4, 0, 1, 69, 19]
1   7170303     [104, 4, 1, 4, 1, 0, 0]      [33, 12, 2, 4, 33, 3, 2]
2  31778885       [0, 1, 3, 1, 3, 1, 1]         [1, 1, 1, 1, 0, 1, 0]
3   2815718    [22, 22, 26, 4, 4, 7, 7]     [12, 12, 13, 6, 7, 9, 12]
4  12134261   [353, 33, 4, 6, 0, 5, 18]  [51, 16, 16, 22, 24, 12, 10]


In [8]:
# Likes: actor and subject vectors using DuckDB
likes_path = "../../data/ale_simplicistic_model/absolute/filtered/likes.parquet"

# Actor vectors
likes_actor_vec = make_vec_duckdb(likes_path, left_on='did_id', vec_name='likes_actor_vec')
con.register("likes_actor_vec", likes_actor_vec)
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(lav.likes_actor_vec, [0,0,0,0,0,0,0]) AS likes_actor_vec
    FROM uoi_activity ua
    LEFT JOIN likes_actor_vec lav ON ua.did_id = lav.did_id
""")

# Subject vectors
likes_subject_vec = make_vec_duckdb(likes_path, left_on='subject_did_id', vec_name='likes_subject_vec')
con.register("likes_subject_vec", likes_subject_vec)
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(lsv.likes_subject_vec, [0,0,0,0,0,0,0]) AS likes_subject_vec
    FROM uoi_activity ua
    LEFT JOIN likes_subject_vec lsv ON ua.did_id = lsv.did_id
""")

print("Likes processed:")
print(con.execute("SELECT did_id, likes_actor_vec, likes_subject_vec FROM uoi_activity LIMIT 5").df())

Likes processed:
     did_id               likes_actor_vec            likes_subject_vec
0   5323188     [66, 35, 20, 5, 8, 10, 6]     [14, 21, 16, 4, 1, 1, 2]
1   1877907       [1, 0, 4, 25, 21, 6, 2]        [0, 0, 0, 1, 0, 0, 0]
2   3157815         [5, 7, 6, 8, 5, 6, 9]  [12, 14, 8, 13, 12, 12, 11]
3  21405059  [12, 27, 16, 31, 15, 30, 15]        [2, 1, 0, 2, 2, 1, 0]
4   3933012   [24, 27, 12, 18, 31, 8, 23]        [4, 6, 1, 2, 6, 1, 3]


## Save the file

In [18]:
# Export final result to parquet using DuckDB
save_path = "../../data/ale_simplicistic_model/absolute/processed/user_activity.parquet"

# Get final dataframe and save
final_df = con.execute("SELECT * FROM uoi_activity").df()

# Convert to pyarrow and save (DuckDB list types are compatible with pyarrow)
arrow_table = pa.Table.from_pandas(final_df, preserve_index=False)
pq.write_table(arrow_table, save_path, compression='zstd')

print('Saved user_table to', save_path)
print('Number of rows:', len(final_df))
print('Columns:', list(final_df.columns))
print('\nSample of final data:')
print(final_df.head())

# Close DuckDB connection
con.close()

Saved user_table to ../../data/ale_simplicistic_model/absolute/processed/user_activity.parquet
Number of rows: 100000
Columns: ['did_id', 'posts_vec', 'blocks_actor_vec', 'blocks_subject_vec', 'follows_actor_vec', 'follows_subject_vec', 'likes_actor_vec', 'likes_subject_vec']

Sample of final data:
     did_id                     posts_vec       blocks_actor_vec  \
0   7708163         [1, 0, 1, 2, 1, 6, 2]  [0, 0, 1, 0, 0, 0, 0]   
1  13815553         [2, 4, 0, 2, 0, 0, 0]  [0, 0, 1, 0, 0, 0, 0]   
2  23661863        [5, 7, 6, 1, 4, 11, 2]  [1, 0, 0, 0, 0, 0, 0]   
3  22010143         [0, 0, 0, 0, 0, 0, 0]  [0, 0, 0, 0, 0, 0, 0]   
4  20796986  [17, 23, 21, 22, 15, 12, 31]  [1, 0, 0, 0, 0, 0, 0]   

      blocks_subject_vec       follows_actor_vec     follows_subject_vec  \
0  [0, 0, 0, 0, 0, 0, 1]   [1, 0, 1, 0, 0, 1, 0]   [0, 0, 0, 0, 0, 0, 0]   
1  [0, 0, 0, 0, 0, 0, 0]   [1, 3, 0, 0, 0, 0, 0]   [1, 0, 0, 1, 0, 0, 0]   
2  [0, 0, 0, 0, 0, 0, 0]   [0, 0, 0, 0, 0, 0, 1]   [0, 1, 0, 0,